In [ ]:
!pip install scikit-learn matplotlib seaborn pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
from snowflake.snowpark.context import get_active_session

# Establish connection to the current Snowflake context
session = get_active_session()

In [ ]:
ml_query = """
WITH CustomerAggregates AS (
    SELECT 
        o.CUSTOMER_ID AS CUSTOMER_KEY,
        MIN(o.ORDER_DATE) AS FIRST_PURCHASE,
        MAX(o.ORDER_DATE) AS LAST_PURCHASE,
        COUNT(DISTINCT o.ORDER_ID) AS TOTAL_ORDERS,
        SUM(o.TOTAL_AMOUNT) AS TOTAL_SPEND,
        AVG(o.TOTAL_AMOUNT) AS AVG_ORDER_VALUE,
        DATEDIFF('day', MAX(o.ORDER_DATE), CURRENT_DATE()) AS DAYS_SINCE_LAST_PURCHASE,
        DATEDIFF('day', MIN(o.ORDER_DATE), CURRENT_DATE()) AS ACQUISITION_TENURE
    FROM ECOMMERCE.ECOMMERCE_SCHEMA.ORDERS o
    GROUP BY o.CUSTOMER_ID
),
RiskAggregates AS (
    SELECT 
        -- Fallback check to capture total late fee exposure from your dynamic table
        dt.CUSTOMER_KEY,
        SUM(dt.TOTAL_LATE_FEES_LEVIED) AS TOTAL_LATE_FEES
    FROM ECOMMERCE.ECOMMERCE_DN_SCHEMA.DT_FINANCIAL_RISK_AUDIT dt
    GROUP BY dt.CUSTOMER_KEY
)
SELECT 
    c.TOTAL_ORDERS,
    c.TOTAL_SPEND,
    c.AVG_ORDER_VALUE,
    c.DAYS_SINCE_LAST_PURCHASE,
    c.ACQUISITION_TENURE,
    COALESCE(r.TOTAL_LATE_FEES, 0.0) AS TOTAL_LATE_FEES,
    -- Target variable: 1 if churned (inactive 365+ days), 0 if active
    CASE WHEN c.DAYS_SINCE_LAST_PURCHASE >= 365 THEN 1 ELSE 0 END AS CHURNED
FROM CustomerAggregates c
LEFT JOIN RiskAggregates r ON c.CUSTOMER_KEY = r.CUSTOMER_KEY;
"""

# Pull data into your notebook DataFrame
df_ml = session.sql(ml_query).to_pandas()

# Preview your dataset structure
print(f"Dataset Shape: {df_ml.shape}")
df_ml.head()

In [ ]:
# 1. Define the continuous numerical features to evaluate
numeric_features = ['TOTAL_ORDERS', 'TOTAL_SPEND', 'AVG_ORDER_VALUE', 'ACQUISITION_TENURE', 'TOTAL_LATE_FEES']

print("--- OUTLIER DETECTION (IQR METHOD) ---")
outlier_indices = set()

# 2. Iterate through features and calculate bounds
for col in numeric_features:
    Q1 = df_ml[col].quantile(0.25)
    Q3 = df_ml[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify row violations
    outliers = df_ml[(df_ml[col] < lower_bound) | (df_ml[col] > upper_bound)]
    outlier_indices.update(outliers.index)
    
    print(f"🔹 {col}: Found {len(outliers)} outliers (Bounds: {lower_bound:.2f} to {upper_bound:.2f})")

print(f"\nTotal unique customer rows containing at least one outlier: {len(outlier_indices)} ({(len(outlier_indices)/len(df_ml))*100:.2f}% of data)")

# 3. Plot Boxplots to visualize the distribution spreads
plt.figure(figsize=(14, 6))
for i, col in enumerate(numeric_features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(data=df_ml, y=col, color='#4c4c7d')
    plt.title(f'{col} Spread')
plt.tight_layout()
plt.show()

In [ ]:
# Separate Features (X) and Target Label (y)
X = df_ml.drop(columns=['CHURNED', 'DAYS_SINCE_LAST_PURCHASE'])
y = df_ml['CHURNED']

# Split into train (80%) and test (20%) architectures
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Class distribution in Training Set:\n{y_train.value_counts(normalize=True)}")

In [ ]:
# Initialize Random Forest
rf_classifier = RandomForestClassifier(
    n_estimators=150,       
    max_depth=10,           
    class_weight='balanced',
    random_state=42,
    n_jobs=-1               
)

rf_classifier.fit(X_train, y_train)

# Generate predictions
y_pred = rf_classifier.predict(X_test)
y_prob = rf_classifier.predict_proba(X_test)[:, 1]

In [ ]:
print("--- MODEL PERFORMANCE EVALUATION ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

# Visualizing Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Active', 'Predicted Churned'], yticklabels=['Actual Active', 'Actual Churned'])
plt.title('Confusion Matrix Breakdown')
plt.show()

In [ ]:
# Extract feature importance metrics
importances = rf_classifier.feature_importances_
indices = np.argsort(importances)[::-1]

# Plot the variables
plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices], y=X.columns[indices], palette="viridis")
plt.title("What Variables Drive Customer Churn Risk?")
plt.xlabel("Relative Importance Score")
plt.ylabel("Customer Feature Attribute")
plt.tight_layout()
plt.show()